In [117]:
from sklearn.neighbors import NearestNeighbors
import glob
import pandas as pd
from astropy.cosmology import Planck18
from astropy import units as u
import numpy as np
from astropy.coordinates import SkyCoord
from scipy.stats import norm
import matplotlib.pyplot as plt

In [118]:
model = ['llr', 'xgb']

In [119]:
target = pd.read_csv('../all_sample_20260702.csv')
tg_d = Planck18.comoving_distance(target.paliya_Z).value
tg_ra = (target.paliya_RA).values
tg_dec = (target.paliya_DEC).values

coord = SkyCoord(ra = tg_ra * u.deg, dec = tg_dec * u.deg, distance = tg_d * u.Mpc)
x = coord.cartesian.x.value
y = coord.cartesian.y.value
z = coord.cartesian.z.value
tg_df = pd.DataFrame({'x':x, 'y':y, 'z':z})

In [120]:
np.cumsum([1, 2, 3,4,5])

array([ 1,  3,  6, 10, 15])

In [135]:
index = np.zeros([len(tg_df)])
index_filter = np.zeros([len(tg_df)])
index_df = pd.DataFrame()
for m in model:
    for i in range(len(tg_df)):
        file_list = glob.glob(f'../data/catalog/ps1/{m}/result/t{tg_ra[i]:08.4f}{tg_dec[i]:+07.4f}.csv')
        file_filtered_list = glob.glob(f'../data/catalog/ps1/{m}/result_filtered/t{tg_ra[i]:08.4f}{tg_dec[i]:+07.4f}.csv')
        df = pd.read_csv(file_list[0])
        surround_z = (df.z_phot).values
        surround_err = (df.z_phot_err).values
        surround_scale = Planck18.comoving_distance(surround_z + surround_err) - Planck18.comoving_distance(surround_z - surround_err)
        surround_d = (df.D_L_Mpc).values
        surround_odds = np.abs(norm.cdf(tg_d[i] - (0.5*u.Mpc).value, loc =surround_d, scale = surround_scale) - norm.cdf(tg_d[i] + (0.5*u.Mpc).value, loc = surround_d, scale = surround_scale))
        surround_ra = (df.raMean).values
        surround_dec = (df.decMean).values

        coord = SkyCoord(ra =surround_ra * u.deg, dec = surround_dec * u.deg, distance = surround_d * u.Mpc)
        x = coord.cartesian.x.value
        y = coord.cartesian.y.value
        z = coord.cartesian.z.value

        df = pd.DataFrame({'x':x, 'y':y, 'z':z})
        nn = NearestNeighbors(n_neighbors= len(df), algorithm= 'auto')
        nn.fit(df)
        neighbors = nn.kneighbors(tg_df.loc[[i]])
        k_index = np.cumsum([surround_odds[i] for i in neighbors[1]])
        k_index = np.where(k_index >= 5)[0]
        try:
            index[i] = neighbors[1][k_index]
        except:
            print(f'{tg_ra[i]}, {tg_dec[i]} don\'t have enough data')
            index[i] = -999

        # df = pd.read_csv(file_filtered_list[0])
        # surround_z = (df.z_phot).values
        # surround_err = (df.z_phot_err).values
        # try:
        #     surround_scale = Planck18.comoving_distance(surround_z + surround_err) - Planck18.comoving_distance(surround_z - surround_err)
        #     surround_d = (df.D_L_Mpc).values
        #     surround_odds = np.abs(norm.cdf(tg_d[i] - (0.5*u.Mpc).value, loc =surround_d, scale = surround_scale) - norm.cdf(tg_d[i] + (0.5*u.Mpc).value, loc = surround_d, scale = surround_scale))
        #     surround_ra = (df.raMean).values
        #     surround_dec = (df.decMean).values
        #     coord = SkyCoord(ra =surround_ra * u.deg, dec = surround_dec * u.deg, distance = surround_d * u.Mpc)
        #     x = coord.cartesian.x.value
        #     y = coord.cartesian.y.value
        #     z = coord.cartesian.z.value
        #     df = pd.DataFrame({'x':x, 'y':y, 'z':z})
        # except:
        #     print(f'{tg_ra[i]}, {tg_dec[i]} don\'t have enough data')
        #     index_filter[i] = -999
        #     continue
        # try:
        #     nn = NearestNeighbors(n_neighbors= k, algorithm= 'auto')
        #     nn.fit(df)
        #     neighbors = nn.kneighbors(tg_df.loc[[i]])
        #     k_index = np.cumsum([surround_odds[i] for i in neighbors[1]])
        #     k_index = np.where(k_index >= 5)[0]
        #     index_filter[i] = neighbors[1][k_index]
        # except:
        #     print(f'{tg_ra[i]}, {tg_dec[i]} don\'t have enough data')
        #     index_filter[i] = -999
    index_df[f'index_{m}'] = [index]
    # index_df[f'index_filtered_{m}'] = [index_filter]

0.4368958, 11.16667 don't have enough data
1.417268, 0.318698 don't have enough data
4.24354, 7.345316 don't have enough data
4.264051, -4.789808 don't have enough data
4.718625, 1.132922 don't have enough data
5.030144, 32.7363 don't have enough data
5.687168, -3.058511 don't have enough data
6.079566, -1.638201 don't have enough data
6.190406, 8.349175 don't have enough data
7.045126, 15.93379 don't have enough data
8.087626, 24.39981 don't have enough data
8.159169, -1.009781 don't have enough data
9.155493, 26.33139 don't have enough data
9.193546, 14.99357 don't have enough data
10.1476, 18.27801 don't have enough data
11.53859, 10.14349 don't have enough data
11.83082, 14.7035 don't have enough data
16.03817, 0.1454773 don't have enough data
16.80014, 14.14583 don't have enough data
17.53756, -10.14539 don't have enough data
18.37521, 4.464441 don't have enough data
18.70282, -0.4961343 don't have enough data
19.31525, 25.44974 don't have enough data
19.50434, -0.496014 don't hav

In [144]:
surround_err

array([0.12088744, 0.14995594, 0.06479306, 0.11365877, 0.2172007 ,
       0.16069672, 0.12575933, 0.05364771, 0.06634144, 0.05799168,
       0.08760368, 0.07270906, 0.14406238, 0.18206719, 0.05839771,
       0.04668345, 0.05022725, 0.14579791, 0.02751443, 0.09217812,
       0.07633244, 0.1496573 , 0.1275317 , 0.13014984, 0.1674002 ,
       0.13894494, 0.02021233, 0.17840293, 0.20316558, 0.10738058,
       0.03044864, 0.12540539, 0.12832092, 0.1849086 , 0.12666339,
       0.11800289, 0.1244927 , 0.05261374, 0.05902686, 0.03817053,
       0.07457901, 0.14799492, 0.17651084, 0.12751757, 0.15903297,
       0.15598224, 0.02578186, 0.02501484, 0.14936727, 0.07350071,
       0.04694721, 0.03497895, 0.17020623, 0.18332   , 0.06031678,
       0.02629019, 0.08434751, 0.05301336, 0.17216569, 0.10976307])

In [123]:
id = index_df.loc[0][f'index_{model[0]}']
id_filtered = index_df.loc[0][f'index_filtered_{model[0]}']
if id != -999:
    file_list = glob.glob(f'../data/catalog/ps1/{model[0]}/result/t{tg_df.loc[0].ra:08.4f}{tg_df.loc[0].dec:+07.4f}.csv')
    df = pd.read_csv(file_list[0])
    surround_d = (df.D_L_Mpc).values
    surround_ang_d = (df.D_L_Mpc).values
    df = pd.DataFrame({'ang_d':surround_ang_d, 'd':surround_d})
    df = df.loc[id]


if id_filtered != -999:
    file_filtered_list = glob.glob(f'../data/catalog/ps1/{model[0]}/result_filtered/t{tg_df.loc[0].ra:08.4f}{tg_df.loc[0].dec:+07.4f}.csv')
    df = pd.read_csv(file_filtered_list[0])
    surround_d = (df.D_L_Mpc).values    
    surround_ra = (df.raMean).values
    surround_dec = (df.decMean).values
    df = pd.DataFrame({'ra':surround_ra, 'dec':surround_dec, 'd':surround_d})
    df = df.loc[id_filtered]

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()